# Day 04 · 視覺化戰情室：Runtime、Web UI 與 Visual Builder

> 第一部・新兵入伍　|　💻 以 CLI／子程序實跑

**前置需求**：🔑 需要 Gemini API 金鑰、💻 會用到終端機／子程序

**對應文章**：`Day 04 - 視覺化戰情室：Runtime、Web UI 與 Visual Builder.md`

## 今天要學會

1. 分辨 `adk run` / `adk web` / `adk api_server` 的適用場合
2. 用 `--save_session` / `--session_id` / `--resume` 三個**互斥**選項
3. 把 session 存進 SQLite 並跨程序讀回

> 原文零 Python。本日把每個 CLI 選項**實際跑一次**並擷取輸出。

## 環境設定

每一天都是獨立的，這段設定刻意重複，讓你可以從任何一天開始。

In [1]:
import warnings

warnings.filterwarnings("ignore")

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared import ask, get_model, new_session, peek_state, print_state, quiet, run_once

quiet()

import google.adk

print("google-adk", google.adk.__version__)

google-adk 2.8.0


## 1. 四種跑法一張表

| 指令 | 介面 | 適合 | 注意 |
|---|---|---|---|
| `adk run` | 終端機 | 快速測試、接 CI（讀 stdin） | 互動式，但可用管線 |
| `adk web` | 瀏覽器 | **開發時最有用**：看事件、state、trace | ⚠️ 官方明示**僅供開發** |
| `adk api_server` | HTTP | 給前端接、事件觸發 | Day 24 會再用到 |
| `adk deploy` | — | 上線 | Day 30 |

## 2. 先準備一個測試用 agent

In [2]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

ADK = Path(sys.executable).parent / "adk"
WORK = Path.cwd() / "_day04"
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir()

api_key = os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY", "")

# `adk run` 是另一個程序，讀 agent.py 裡寫死的模型 ID。免費層配額是每個模型
# 分開算的，先挑一個還有額度的再寫進去。
from shared import pick_available_model

MODEL = pick_available_model()
print(f"本日使用模型：{MODEL}\n")

AGENT = WORK / "note_agent"
AGENT.mkdir()
(AGENT / "__init__.py").write_text("from . import agent\n", encoding="utf-8")
(AGENT / "agent.py").write_text('''"""一個會把資訊記進 state 的 agent，方便觀察 session。"""

from google.adk.agents import LlmAgent
from google.adk.tools import ToolContext


def remember(key: str, value: str, tool_context: ToolContext) -> dict:
    """記住一項資訊。

    Args:
        key: 項目名稱。
        value: 內容。
    """
    tool_context.state[key] = value
    return {"ok": True, "remembered": {key: value}}


def recall(key: str, tool_context: ToolContext) -> dict:
    """回想先前記住的資訊。

    Args:
        key: 項目名稱。
    """
    return {"key": key, "value": tool_context.state.get(key, "（沒有記錄）")}


root_agent = LlmAgent(
    name="note_agent",
    model="__MODEL__",
    description="幫使用者記住與回想事情。",
    instruction=(
        "你是記事助理。使用者要你記住什麼就呼叫 remember，"
        "要你回想就呼叫 recall。用繁體中文簡短回覆。"
    ),
    tools=[remember, recall],
)
'''.replace("__MODEL__", MODEL), encoding="utf-8")
(AGENT / ".env").write_text(f"GOOGLE_API_KEY={api_key}\n", encoding="utf-8")

print("建立完成:")
for p in sorted(AGENT.iterdir()):
    print(f"  {p.name}")


def run(cmd, cwd=WORK, stdin=None, timeout=200, quiet_ok=False):
    r = subprocess.run([str(c) for c in cmd], cwd=cwd, input=stdin,
                       capture_output=True, text=True, timeout=timeout)
    print(f"$ {' '.join(str(c) for c in cmd)}")
    if r.stdout.strip():
        print(r.stdout.rstrip())
    if r.returncode != 0 and not quiet_ok:
        print(f"--- exit {r.returncode} / stderr ---")
        print((r.stderr or "").rstrip()[-1000:])
    return r

⏭️  gemini-flash-lite-latest 跳過：配額用完 (429)


⏭️  gemini-3.5-flash-lite 跳過：配額用完 (429)


✅ gemini-3.1-flash-lite 可用
本日使用模型：gemini-3.1-flash-lite

建立完成:
  .env
  __init__.py
  agent.py


## 3. `adk run` 的選項

In [3]:
run([ADK, "run", "--help"])

$ /Users/linshihuan/Dev/github/adk_tutor/.venv/bin/adk run --help
Usage: adk run [OPTIONS] AGENT [QUERY]

  Runs an agent. If no query is provided, enters interactive mode.

  AGENT: The path to the agent source code folder. QUERY: Optional. The user
  message to send to the agent for a single-step run.

  Example:

    adk run path/to/my_agent   adk run path/to/my_agent "hello"

Options:
  --enable_features TEXT          Optional. Comma-separated list of feature
                                  names to enable. This provides an
                                  alternative to environment variables for
                                  enabling experimental features. Example: --e
                                  nable_features=JSON_SCHEMA_FOR_FUNC_DECL,PRO
                                  GRESSIVE_SSE_STREAMING
  --disable_features TEXT         Optional. Comma-separated list of feature
                                  names to disable. This provides an
                             

CompletedProcess(args=['/Users/linshihuan/Dev/github/adk_tutor/.venv/bin/adk', 'run', '--help'], returncode=0, stdout='Usage: adk run [OPTIONS] AGENT [QUERY]\n\n  Runs an agent. If no query is provided, enters interactive mode.\n\n  AGENT: The path to the agent source code folder. QUERY: Optional. The user\n  message to send to the agent for a single-step run.\n\n  Example:\n\n    adk run path/to/my_agent   adk run path/to/my_agent "hello"\n\nOptions:\n  --enable_features TEXT          Optional. Comma-separated list of feature\n                                  names to enable. This provides an\n                                  alternative to environment variables for\n                                  enabling experimental features. Example: --e\n                                  nable_features=JSON_SCHEMA_FOR_FUNC_DECL,PRO\n                                  GRESSIVE_SSE_STREAMING\n  --disable_features TEXT         Optional. Comma-separated list of feature\n                          

### 基本執行

In [4]:
run([ADK, "run", "note_agent"], stdin="記住我的專案代號是 Falcon\nexit\n")

$ /Users/linshihuan/Dev/github/adk_tutor/.venv/bin/adk run note_agent
Log setup complete: /tmp/agents_log/agent.20260904_020648.log
To access latest log: tail -F /tmp/agents_log/agent.latest.log
Running agent note_agent, type exit to exit.
[user]: [note_agent]: 好的，已記住您的專案代號為 Falcon。
[user]:


CompletedProcess(args=['/Users/linshihuan/Dev/github/adk_tutor/.venv/bin/adk', 'run', 'note_agent'], returncode=0, stdout='Log setup complete: /tmp/agents_log/agent.20260904_020648.log\nTo access latest log: tail -F /tmp/agents_log/agent.latest.log\nRunning agent note_agent, type exit to exit.\n[user]: [note_agent]: 好的，已記住您的專案代號為 Falcon。\n[user]: ', stderr='/Users/linshihuan/Dev/github/adk_tutor/.venv/lib/python3.13/site-packages/google/adk/cli/cli.py:335: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.\n  credential_service = InMemoryCredentialService()\n/Users/linshihuan/Dev/github/adk_tutor/.venv/lib/python3.13/site-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It

## 4. `--save_session`：把對話存下來

In [5]:
r = run(
    [ADK, "run", "note_agent", "--save_session", "--session_id", "demo-1"],
    stdin="記住我最喜歡的語言是 Rust\nexit\n",
)

print("\n--- 產生的檔案 ---")
for p in sorted(WORK.rglob("*.json")):
    print(f"  {p.relative_to(WORK)}  ({p.stat().st_size} bytes)")

$ /Users/linshihuan/Dev/github/adk_tutor/.venv/bin/adk run note_agent --save_session --session_id demo-1
Log setup complete: /tmp/agents_log/agent.20260904_020656.log
To access latest log: tail -F /tmp/agents_log/agent.latest.log
Running agent note_agent, type exit to exit.
[user]: [note_agent]: 好的，我已經記住您最喜歡的語言是 Rust。
[user]: Session saved to /Users/linshihuan/Dev/github/adk_tutor/30day_practice/day04_runtime_web_ui/_day04/note_agent/demo-1.session.json

--- 產生的檔案 ---
  note_agent/demo-1.session.json  (4079 bytes)


In [6]:
import json

session_files = list(WORK.rglob("*.session.json")) or list(WORK.rglob("*.json"))
if session_files:
    data = json.loads(session_files[0].read_text(encoding="utf-8"))
    print("session 檔的頂層欄位:", list(data))
    print("\nstate:", json.dumps(data.get("state", {}), ensure_ascii=False))
    print(f"events: {len(data.get('events', []))} 筆")
    for ev in data.get("events", [])[:6]:
        author = ev.get("author")
        parts = (ev.get("content") or {}).get("parts") or []
        text = "".join(p.get("text") or "" for p in parts)[:60]
        print(f"  [{author}] {text}")
else:
    print("（沒有找到 session 檔，可能是這個版本的輸出位置不同）")

session 檔的頂層欄位: ['id', 'appName', 'userId', 'state', 'events', 'lastUpdateTime']

state: {"最喜歡的語言": "Rust"}
events: 4 筆
  [user] 記住我最喜歡的語言是 Rust
  [note_agent] 
  [note_agent] 
  [note_agent] 好的，我已經記住您最喜歡的語言是 Rust。


這個 JSON 就是 Day 09 要細講的 `Session` 物件序列化之後的樣子：
**`state`（現在怎樣）+ `events`（發生過什麼）**。

## 5. `--resume`：接續之前的對話

In [7]:
if session_files:
    r = run(
        [ADK, "run", "note_agent", "--resume", str(session_files[0])],
        stdin="我最喜歡的語言是什麼？\nexit\n",
    )
else:
    print("（略過：沒有可恢復的 session 檔）")

$ /Users/linshihuan/Dev/github/adk_tutor/.venv/bin/adk run note_agent --resume /Users/linshihuan/Dev/github/adk_tutor/30day_practice/day04_runtime_web_ui/_day04/note_agent/demo-1.session.json
Log setup complete: /tmp/agents_log/agent.20260904_020702.log
To access latest log: tail -F /tmp/agents_log/agent.latest.log
[user]: 記住我最喜歡的語言是 Rust
[note_agent]: 好的，我已經記住您最喜歡的語言是 Rust。
[user]: [note_agent]: 您最喜歡的語言是 Rust。
[user]:


## 6. 📌 補充：「三個 session 選項互斥」實測

文件說 `--save_session` / `--session_id` / `--resume` 是互斥的。
那同時給兩個會怎樣？直覺會以為是「參數錯誤，直接退出」。實測一下：

In [8]:
if session_files:
    r = run(
        [ADK, "run", "note_agent",
         "--resume", str(session_files[0]),
         "--session_id", "another-one"],
        stdin="exit\n", quiet_ok=True,
    )
    print(f"\nexit code: {r.returncode}  ← 0 代表它沒有把這當成錯誤")
    err = (r.stderr or r.stdout).strip().splitlines()
    for ln in err[-5:]:
        print("  ", ln[:160])

$ /Users/linshihuan/Dev/github/adk_tutor/.venv/bin/adk run note_agent --resume /Users/linshihuan/Dev/github/adk_tutor/30day_practice/day04_runtime_web_ui/_day04/note_agent/demo-1.session.json --session_id another-one
Log setup complete: /tmp/agents_log/agent.20260904_020710.log
To access latest log: tail -F /tmp/agents_log/agent.latest.log
[user]: 記住我最喜歡的語言是 Rust
[note_agent]: 好的，我已經記住您最喜歡的語言是 Rust。
[user]:

exit code: 0  ← 0 代表它沒有把這當成錯誤
   /Users/linshihuan/Dev/github/adk_tutor/.venv/lib/python3.13/site-packages/google/adk/cli/cli.py:335: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This
     credential_service = InMemoryCredentialService()
   /Users/linshihuan/Dev/github/adk_tutor/.venv/lib/python3.13/site-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [E
     super().__init__()


### 結果：它不會報錯

`exit code = 0`，指令正常結束。ADK **沒有**把「同時給兩個 session 選項」
當成參數錯誤，而是**安靜地採用其中一個、忽略另一個**——
從輸出可以看出來，它讀了 `--resume` 的檔案，`--session_id another-one` 被丟掉了。

**這比直接報錯更危險。** 想像一下：

- 你在腳本裡同時傳了 `--session_id`，以為對話會寫進那個 id
- 實際上它接續了 `--resume` 的舊 session
- 沒有任何警告，你要等到查資料時才發現 session 對不上

**實務建議**：這三個選項在你的腳本裡只出現一個，
而且用完之後**驗證 session id 真的是你要的那個**，不要假設參數有被採納。

## 7. `--session_service_uri`：改用 SQLite

預設 session 存在記憶體，程序結束就沒了。指定 `--session_service_uri`
就能換成持久化後端——**agent 的程式碼一行都不用改**。

In [9]:
db_path = WORK / "sessions.db"
uri = f"sqlite:///{db_path}"

run([ADK, "run", "note_agent", "--session_service_uri", uri,
     "--session_id", "sql-1", "--save_session"],
    stdin="記住我住在台北\nexit\n")

print(f"\n資料庫檔案存在: {db_path.exists()}", end="")
if db_path.exists():
    print(f"（{db_path.stat().st_size} bytes）")

$ /Users/linshihuan/Dev/github/adk_tutor/.venv/bin/adk run note_agent --session_service_uri sqlite:////Users/linshihuan/Dev/github/adk_tutor/30day_practice/day04_runtime_web_ui/_day04/sessions.db --session_id sql-1 --save_session
Log setup complete: /tmp/agents_log/agent.20260904_020712.log
To access latest log: tail -F /tmp/agents_log/agent.latest.log
Running agent note_agent, type exit to exit.
[user]: [note_agent]: 好的，我記住您住在台北了。
[user]: Session saved to /Users/linshihuan/Dev/github/adk_tutor/30day_practice/day04_runtime_web_ui/_day04/note_agent/sql-1.session.json

資料庫檔案存在: True（36864 bytes）


In [10]:
# 換一個全新的程序，用同一個 DB 讀回來
if db_path.exists():
    run([ADK, "run", "note_agent", "--session_service_uri", uri,
         "--session_id", "sql-1"],
        stdin="我住在哪裡？\nexit\n")

$ /Users/linshihuan/Dev/github/adk_tutor/.venv/bin/adk run note_agent --session_service_uri sqlite:////Users/linshihuan/Dev/github/adk_tutor/30day_practice/day04_runtime_web_ui/_day04/sessions.db --session_id sql-1
Log setup complete: /tmp/agents_log/agent.20260904_020723.log
To access latest log: tail -F /tmp/agents_log/agent.latest.log
Running agent note_agent, type exit to exit.
[user]: [note_agent]: 您目前還沒告訴我您住在哪裡喔。如果您想讓我記住，可以跟我說：「記住我住在 [地址/地方]」。
[user]:


跨程序記得住。這就是 Day 09 的 `DatabaseSessionService` 在 CLI 上的樣子。

In [11]:
import sqlite3

if db_path.exists():
    con = sqlite3.connect(db_path)
    tables = [r[0] for r in con.execute(
        "SELECT name FROM sqlite_master WHERE type='table'")]
    print("SQLite 裡的表:", tables)
    for t in tables:
        n = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
        print(f"  {t}: {n} 筆")
    con.close()

SQLite 裡的表: ['app_states', 'user_states', 'sessions', 'events']
  app_states: 0 筆
  user_states: 0 筆
  sessions: 2 筆
  events: 8 筆


## 8. `adk web` 與 `adk api_server`

這兩個是長駐服務，notebook 裡不方便跑。先看選項：

In [12]:
run([ADK, "web", "--help"])

$ /Users/linshihuan/Dev/github/adk_tutor/.venv/bin/adk web --help
Usage: adk web [OPTIONS] [AGENTS_DIR]

  Starts a FastAPI server with Web UI for agents.

  AGENTS_DIR: The directory of agents (where each subdirectory is a single
  agent containing `agent.py`, `__init__.py`, or `root_agent.yaml`) or a path
  pointing directly to a single agent folder.

  This server is intended for local development. Its endpoints are
  unauthenticated, so run it on a trusted network only and do not expose it to
  untrusted or public networks.

  Example:

    adk web --session_service_uri=[uri] --port=[port] path/to/agents_dir

Options:
  --enable_features TEXT          Optional. Comma-separated list of feature
                                  names to enable. This provides an
                                  alternative to environment variables for
                                  enabling experimental features. Example: --e
                                  nable_features=JSON_SCHEMA_FOR_FUNC_DE

CompletedProcess(args=['/Users/linshihuan/Dev/github/adk_tutor/.venv/bin/adk', 'web', '--help'], returncode=0, stdout="Usage: adk web [OPTIONS] [AGENTS_DIR]\n\n  Starts a FastAPI server with Web UI for agents.\n\n  AGENTS_DIR: The directory of agents (where each subdirectory is a single\n  agent containing `agent.py`, `__init__.py`, or `root_agent.yaml`) or a path\n  pointing directly to a single agent folder.\n\n  This server is intended for local development. Its endpoints are\n  unauthenticated, so run it on a trusted network only and do not expose it to\n  untrusted or public networks.\n\n  Example:\n\n    adk web --session_service_uri=[uri] --port=[port] path/to/agents_dir\n\nOptions:\n  --enable_features TEXT          Optional. Comma-separated list of feature\n                                  names to enable. This provides an\n                                  alternative to environment variables for\n                                  enabling experimental features. Example: --e

### 自己動手開起來

```bash
cd 30day_practice/day04_runtime_web_ui/_day04   # 本日建立的目錄
adk web
# 打開 http://localhost:8000
```

Dev UI 能看到的東西，正好對應我們前面手動印的：

| Dev UI 分頁 | 相當於 |
|---|---|
| Events | `trace=True` 印的事件串流 |
| State | `peek_state()` |
| Trace | 每一步的耗時與 token |
| Eval | Day 26 的評估 |

> ⚠️ **官方 Caution：ADK Web 僅供開發使用。**
> 它沒有認證、沒有多租戶隔離，不要對外開放。

### `adk api_server` 快速確認

In [13]:
run([ADK, "api_server", "--help"], quiet_ok=True)

$ /Users/linshihuan/Dev/github/adk_tutor/.venv/bin/adk api_server --help
Usage: adk api_server [OPTIONS] [AGENTS_DIR]

  Starts a FastAPI server for agents.

  AGENTS_DIR: The directory of agents (where each subdirectory is a single
  agent containing `agent.py`, `__init__.py`, or `root_agent.yaml`) or a path
  pointing directly to a single agent folder.

  This server's endpoints are unauthenticated. Run it on a trusted network
  only, and put it behind your own authentication and authorization layer
  before exposing it to untrusted or public networks or serving multiple
  users.

  Example:

    adk api_server --session_service_uri=[uri] --port=[port]
    path/to/agents_dir

Options:
  --enable_features TEXT          Optional. Comma-separated list of feature
                                  names to enable. This provides an
                                  alternative to environment variables for
                                  enabling experimental features. Example: --e
      

CompletedProcess(args=['/Users/linshihuan/Dev/github/adk_tutor/.venv/bin/adk', 'api_server', '--help'], returncode=0, stdout="Usage: adk api_server [OPTIONS] [AGENTS_DIR]\n\n  Starts a FastAPI server for agents.\n\n  AGENTS_DIR: The directory of agents (where each subdirectory is a single\n  agent containing `agent.py`, `__init__.py`, or `root_agent.yaml`) or a path\n  pointing directly to a single agent folder.\n\n  This server's endpoints are unauthenticated. Run it on a trusted network\n  only, and put it behind your own authentication and authorization layer\n  before exposing it to untrusted or public networks or serving multiple\n  users.\n\n  Example:\n\n    adk api_server --session_service_uri=[uri] --port=[port]\n    path/to/agents_dir\n\nOptions:\n  --enable_features TEXT          Optional. Comma-separated list of feature\n                                  names to enable. This provides an\n                                  alternative to environment variables for\n          

## 9. Visual Builder

ADK Python v1.18.0 起有 Visual Builder（Experimental），可以用拖拉的方式建 agent。

**一個很重要的限制**：它只能重新編輯**由它自己建立**的 agent。
你手寫的 `agent.py` 匯不進去。所以它適合原型，不適合接手既有專案。

In [14]:
shutil.rmtree(WORK, ignore_errors=True)
print("已清理工作目錄")

已清理工作目錄


## 10. 常見錯誤與踩坑

| 症狀 | 原因 |
|---|---|
| `--resume` 和 `--session_id` 一起給 | **不會報錯**，其中一個被安靜忽略——比報錯更難查 |
| 換了 terminal 就忘記之前的對話 | 預設 session 在記憶體。要 `--session_service_uri` |
| `adk web` 打不開 | 要在**包含 agent 目錄的那一層**執行，不是在 agent 目錄裡面 |
| Visual Builder 匯不進既有 agent | 它只能編輯自己建立的 agent |

## 11. 動手練習

1. 用 `--replay` 選項準備一個 JSON（含 `state` 和 `queries`），批次跑多個問題。
2. 把 `--session_service_uri` 指到 `sqlite:///shared.db`，
   然後用 notebook 的 `DatabaseSessionService` 讀同一個檔（Day 09 會用到）。
3. 開 `adk web`，跑一次工具呼叫，比較 Events 分頁跟本日 `trace=True` 的資訊量。

## 本日回顧

- **四種跑法**：`run`（終端機／CI）、`web`（開發 UI）、`api_server`（給前端）、`deploy`（上線）。
- **三個 session 選項互斥，但 ADK 不會替你把關**：同時給兩個時
  `exit code` 仍是 0，其中一個被安靜忽略。腳本裡只放一個，並驗證結果。
- **session 檔就是 `state` + `events`**，跟 Day 09 的 `Session` 物件是同一個東西。
- **`--session_service_uri` 換持久化後端**，agent 程式碼不用改。
- **`adk web` 官方明示僅供開發**；**Visual Builder 只能編輯自己建的 agent**。

---
**下一天 → `../day05_code_with_ai/`**